# Dynamical spectral function $A(k,\omega)$ from bitstring-sampled quantum subspaces — IBM Heron
**Method:** prepare a *number-conserving* $(N{+}1)$-electron reference, Trotter-evolve it shallowly, sample computational-basis bitstrings, recover to the $(N_\alpha{+}1,N_\beta)$ sector (S-CoRe), and build the Lehmann spectral function **classically** in the sampled determinant subspace. No Hadamard tests; the QPU only samples.
Refs: TE-QSCI (arXiv:2412.13839), SKQD (2501.09702), SQD+S-CoRe (2405.05068), nearest sibling (2510.24911). System: 1D Hubbard L=6, U/t=4, 12 qubits — $A(k,\omega)$ is ED-checkable.

In [ ]:
!pip install -q ffsim qiskit qiskit-ibm-runtime qiskit-addon-sqd pyscf numpy scipy matplotlib

### Mount Google Drive (results are saved here so they can be read back)

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import json, os
OUT='/content/drive/MyDrive/spectral_heron_result.json'
def save(d):
    prev={}
    if os.path.exists(OUT):
        try: prev=json.load(open(OUT))
        except: pass
    prev.update(d); json.dump(prev,open(OUT,'w'),indent=1); print('saved ->',OUT)

## 1. Hamiltonian + exact reference spectral function (classical check)

In [ ]:
import numpy as np, scipy.sparse as sp
L=6; U=4.0; thop=1.0; eta=0.15; M=2*L  # spin-orbital p = site*2+spin
def c_op(p,dim):
    r=[];c=[];d=[]
    for s in range(dim):
        if (s>>p)&1:
            sg=(-1)**bin(s&((1<<p)-1)).count('1'); r.append(s&~(1<<p)); c.append(s); d.append(float(sg))
    return sp.csr_matrix((d,(r,c)),shape=(dim,dim))
dim=1<<M; C=[c_op(p,dim) for p in range(M)]; Cd=[c.T.conj() for c in C]
Nocc=np.array([bin(s).count('1') for s in range(dim)])
Szocc=np.array([sum(((s>>(2*i))&1)-((s>>(2*i+1))&1) for i in range(L)) for s in range(dim)])
H=sp.csr_matrix((dim,dim))
for i in range(L):  # periodic chain
    j=(i+1)%L
    for spin in (0,1):
        a=2*i+spin; b=2*j+spin; H=H-thop*(Cd[a]@C[b]+Cd[b]@C[a])
for i in range(L): H=H+U*(Cd[2*i]@C[2*i])@(Cd[2*i+1]@C[2*i+1])
H=H.tocsr()
gi=np.where((Nocc==L)&(Szocc==0))[0]
w,v=np.linalg.eigh(H[gi][:,gi].toarray()); E0=w[0]; psi0=np.zeros(dim,complex); psi0[gi]=v[:,0]
p_orb=0; phi=Cd[2*p_orb]@psi0; si=np.where((Nocc==L+1)&(Szocc==1))[0]
En,Vn=np.linalg.eigh(H[si][:,si].toarray()); coef=Vn.conj().T@phi[si]
grid=np.linspace((En-E0).min()-1,(En-E0).max()+1,600)
def spec(pw,ww):
    A=np.zeros_like(grid)
    for a,b in zip(pw,ww): A+=b*(eta/np.pi)/((grid-a)**2+eta**2)
    return A
A_exact=spec(En-E0,np.abs(coef)**2)
print('exact reference ready; N+1 sector =',len(si))

## 2. Number-conserving (N+1) reference + shallow Trotter circuits (one per Krylov time)
On-site $U$ term $\to$ native fractional **RZZ**; hopping $\to$ Givens ($XX{+}YY$). Keep $n_{trot}$ small: Trotter error only changes *which* configs are sampled, not the variational spectrum.

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, transpile
norb=L; nelec_add=(4,3)   # (N+1): 4 up, 3 down
K=7; n_trot=3; dt_total=0.5/thop
def givens(qc,a,b,theta):   # e^{-i theta (XX+YY)/2} hopping between spin-orbitals a,b
    qc.rxx(theta,a,b); qc.ryy(theta,a,b)
def build_circuit(k):
    q=QuantumRegister(2*L); qc=QuantumCircuit(q)
    for orb in [0,1,2,3]: qc.x(q[2*orb])       # 4 up electrons (even = up)
    for orb in [0,1,2]:  qc.x(q[2*orb+1])      # 3 down electrons (odd = down)
    dt=(k*dt_total)/max(n_trot,1)
    for _ in range(n_trot):
        for i in range(L): qc.rzz(2*U*dt, q[2*i], q[2*i+1])            # on-site U (native RZZ)
        for spin in (0,1):                                            # hopping per spin
            for i in range(L):
                j=(i+1)%L; givens(qc, q[2*i+spin], q[2*j+spin], thop*dt)
    qc.measure_all(); return qc
circuits=[build_circuit(k) for k in range(K)]
print('built',K,'circuits; depth of last:',circuits[-1].depth())

## 3a. LOCAL validation (run this FIRST — no hardware time)
Sample the ideal statevector of each Trotter circuit locally, run S-CoRe-free pooling, and confirm the method reproduces the exact $A(\omega)$ before touching Heron.

In [ ]:
# exact-statevector sampling of the SAME Trotter circuits (local surrogate for the QPU)
from qiskit.quantum_info import Statevector
from collections import Counter
def bitstring_to_config(bs):   # qiskit little-endian -> our Fock integer (spin-orbital p=site*2+spin)
    return int(bs[::-1],2)
seen=set(); shots=20000
for qc in circuits:
    sv=Statevector(qc.remove_final_measurements(inplace=False))
    probs=np.abs(sv.data)**2; idx=np.random.choice(len(probs),size=shots,p=probs/probs.sum())
    for t in np.unique(idx):
        # keep only (N+1)=7 electrons, Sz=+1 configs (post-selection = the S-CoRe-free filter)
        if Nocc[t]==L+1 and Szocc[t]==1: seen.add(int(t))
S=np.array(sorted(seen))
# map sampled Fock configs into the si-sector index space, diagonalize, Lehmann
pos={int(x):i for i,x in enumerate(si)}; Sidx=np.array([pos[x] for x in S if x in pos])
HS=H[si][:,si].toarray()[np.ix_(Sidx,Sidx)]; Em,Um=np.linalg.eigh(HS); aS=Um.conj().T@phi[si][Sidx]
A_local=spec(Em-E0,np.abs(aS)**2)
rel=np.trapezoid(np.abs(A_local-A_exact),grid)/np.trapezoid(A_exact,grid)
print('LOCAL |S|=',len(Sidx),' rel-L1 vs exact =',round(float(rel),4))
save({'local_relL1':float(rel),'local_S':int(len(Sidx)),'grid':grid.tolist(),
      'A_exact':A_exact.tolist(),'A_local':A_local.tolist(),'L':L,'U':U})
import matplotlib.pyplot as plt
plt.plot(grid,A_exact,'k',label='exact'); plt.plot(grid,A_local,'--',label='sampled subspace (local)')
plt.legend(); plt.xlabel('omega-E0'); plt.ylabel('A(omega)'); plt.title('Local validation'); plt.show()

## 3b. IBM Heron credentials (fill your token) and backend

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
# QiskitRuntimeService.save_account(channel='ibm_quantum', token='PASTE_YOUR_IBM_TOKEN', overwrite=True)
service=QiskitRuntimeService()
backend=service.least_busy(operational=True, simulator=False, min_num_qubits=12)
print('backend:',backend.name)

## 4. Run on Heron (TREX readout + Pauli twirl + dynamical decoupling; NO ZNE)

In [ ]:
isa=transpile(circuits, backend=backend, optimization_level=3)
sampler=SamplerV2(backend)
sampler.options.twirling.enable_gates=True
sampler.options.twirling.enable_measure=True      # TREX
sampler.options.dynamical_decoupling.enable=True
job=sampler.run(isa, shots=50000); print('job id:',job.job_id())
res=job.result()

## 5. Pool + S-CoRe to $(N_\alpha{+}1,N_\beta)=(4,3)$, then classical Lehmann $A(\omega)$

In [ ]:
from qiskit_addon_sqd.counts import counts_to_arrays
# pool bitstrings across all Krylov times
allbits=[]
for pub in res:
    d=pub.data; reg=list(d.keys())[0]; c=getattr(d,reg).get_counts()
    for bs,ct in c.items(): allbits += [bs]*ct
# S-CoRe recovery to the (4,3) sector -- see qiskit-addon-sqd docs for recover_configurations();
# minimal version = post-select exact Hamming weights (primary error suppression):
seen=set()
for bs in allbits:
    t=bitstring_to_config(bs)
    if Nocc[t]==L+1 and Szocc[t]==1: seen.add(int(t))
S=np.array(sorted(seen)); Sidx=np.array([pos[x] for x in S if x in pos])
HS=H[si][:,si].toarray()[np.ix_(Sidx,Sidx)]; Em,Um=np.linalg.eigh(HS); aS=Um.conj().T@phi[si][Sidx]
A_hw=spec(Em-E0,np.abs(aS)**2)
rel=np.trapezoid(np.abs(A_hw-A_exact),grid)/np.trapezoid(A_exact,grid)
print('HARDWARE |S|=',len(Sidx),' rel-L1 vs exact =',round(float(rel),4))
save({'hw_relL1':float(rel),'hw_S':int(len(Sidx)),'backend':backend.name,
      'job_id':job.job_id(),'shots':50000,'A_hw':A_hw.tolist()})
plt.plot(grid,A_exact,'k',label='exact'); plt.plot(grid,A_hw,'--',label='Heron sampled subspace')
plt.legend(); plt.xlabel('omega-E0'); plt.ylabel('A(omega)'); plt.title('Spectral function on IBM Heron'); plt.show()

### Notes
- **Validate locally (3a) first.** Only run 3b–5 once the local rel-L1 is small.
- For full S-CoRe (probabilistic bit-flip recovery beyond post-selection) use `qiskit_addon_sqd.configuration_recovery.recover_configurations` with `target=(4,3)` and the sample-batch occupations.
- The entire $c^\dagger_p$ / Lehmann construction is classical; the QPU only samples $\to$ this is the open-gap method: **dynamical spectral functions from purely bitstring-sampled quantum subspaces.**